# ML6402 – Practical 03: Data Pre-processing

**Instructor:** Dr. Ahmed Sabbah  
**Course:** Machine Learning (ML6402)

This practical follows Lecture 3 step by step.

### Main workflow

**Inspect → Clean → Integrate → Transform → Encode → Split → Pipeline**

### Learning outcomes

By the end of this practical, you should be able to:

1. inspect a dataset using core Pandas commands;
2. identify missing values, duplicates, inconsistent categories, and possible outliers;
3. handle missing values appropriately;
4. clean inconsistent text values;
5. perform a simple data-integration example;
6. aggregate and discretize data;
7. distinguish normalization from standardization;
8. encode categorical data appropriately;
9. split data without leakage;
10. build a reproducible preprocessing pipeline.

## Part 1 – Load the dataset

The guided example uses a small dataset with four attributes:

- **City**
- **Age**
- **Salary**
- **Purchased**

### Choose one loading method

This notebook supports:

1. **Local Anaconda / Jupyter**
2. **Google Colab**

Use only one method at a time.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# DATASET LOADING – CHOOSE ONE METHOD ONLY
# ============================================================

# METHOD 1: Local Anaconda / Jupyter
DATA_PATH = Path("data") / "salary.csv"
df = pd.read_csv(DATA_PATH)

# METHOD 2: Google Colab
# Comment Method 1 above and uncomment the following two lines:
# DATA_URL = "https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture03_Data_Preprocessing/data/salary.csv"
# df = pd.read_csv(DATA_URL)

df

## Part 2 – First inspection: never transform before understanding

The first goal is to understand the dataset as it currently exists.

We will use the most important Pandas inspection commands from Lecture 3.

### 2.1 `head()`

`head()` displays the first rows of the DataFrame.

Use it immediately after loading a file to verify that:

- the file loaded correctly;
- the columns are correct;
- values appear in the expected format.

In [ ]:
df.head()

### 2.2 `tail()` and `sample()`

- `tail()` shows the last rows.
- `sample()` shows randomly selected rows.

These are useful when we do not want to inspect only the beginning of a dataset.

In [ ]:
print("Last rows:")
display(df.tail())

print("Random sample:")
display(df.sample(3, random_state=42))

### 2.3 Dataset dimensions with `shape`

`shape` returns:

```
(number_of_rows, number_of_columns)
```

In [ ]:
print("Dataset shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

### 2.4 Column names and data types

`columns` gives the column names.

`dtypes` reports how Pandas currently represents each column.

In [ ]:
print("Columns:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

### 2.5 `info()`

`info()` provides a compact summary of:

- number of rows;
- column names;
- non-null counts;
- data types;
- approximate memory usage.

This is one of the most useful first commands when inspecting a new dataset.

In [ ]:
df.info()

### 2.6 `describe()`

For numerical columns, `describe()` reports statistics such as:

- count
- mean
- standard deviation
- minimum
- quartiles
- maximum

These values help us understand center, spread, scale, and possible unusual values.

In [ ]:
df.describe(numeric_only=True)

## Part 3 – Missing values

Lecture 3 emphasizes that we should inspect missingness **before** deciding how to handle it.

### 3.1 Detect missing values with `isna()`

`isna()` returns a Boolean DataFrame:

- `True` = missing
- `False` = present

In [ ]:
df.isna()

### 3.2 Count missing values

Combining `isna()` with `sum()` gives the number of missing observations in each column.

In [ ]:
df.isna().sum()

### 3.3 Calculate percentage missing

The percentage can be more informative than the count for larger datasets.

In [ ]:
missing_pct = df.isna().mean() * 100

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": missing_pct
})

missing_summary

### Question

Which columns contain missing values?

Would deleting the corresponding rows be a good decision for this very small dataset? Explain.

## Part 4 – Inspect categorical values

Before encoding categories, first inspect the categories that actually exist.

### 4.1 `value_counts()`

This command helps reveal:

- common categories;
- rare categories;
- misspellings;
- unexpected representations.

In [ ]:
print("City values:")
print(df["City"].value_counts(dropna=False))

print("\nPurchased values:")
print(df["Purchased"].value_counts(dropna=False))

### 4.2 `unique()` and `nunique()`

- `unique()` shows the distinct category values.
- `nunique()` counts them.

In [ ]:
print("Unique cities:", df["City"].unique())
print("Number of unique cities:", df["City"].nunique())

## Part 5 – Duplicate records

`duplicated()` identifies rows that are exact duplicates.

In [ ]:
print("Duplicate mask:")
print(df.duplicated())

print("\nNumber of duplicated rows:")
print(df.duplicated().sum())

The original small dataset may contain no duplicates. To understand the command clearly, we create a temporary copy containing one deliberate duplicate.

In [ ]:
duplicate_demo = pd.concat([df, df.iloc[[0]]], ignore_index=True)

print("Rows before:", len(df))
print("Rows after adding duplicate:", len(duplicate_demo))
print("Detected duplicates:", duplicate_demo.duplicated().sum())

duplicate_demo.tail()

### Removing duplicates

`drop_duplicates()` removes duplicated rows.

In [ ]:
duplicate_demo_clean = duplicate_demo.drop_duplicates()

print("Rows after removing duplicates:", len(duplicate_demo_clean))

## Part 6 – Inconsistent categorical values

A category can represent the same concept in several inconsistent forms:

- `Mumbai`
- `mumbai`
- ` Mumbai `

We create a temporary example to practice detecting and fixing this problem.

In [ ]:
messy_demo = df.copy()

messy_demo.loc[0, "City"] = " Hyderabad "
messy_demo.loc[1, "City"] = "mumbai"

print("Before cleaning:")
print(messy_demo["City"].value_counts(dropna=False))

### Clean strings with `str.strip()` and `str.title()`

- `str.strip()` removes leading/trailing spaces.
- `str.title()` standardizes capitalization.

In [ ]:
messy_demo["City"] = messy_demo["City"].str.strip().str.title()

print("After cleaning:")
print(messy_demo["City"].value_counts(dropna=False))

## Part 7 – Missing-value strategies

Possible strategies include:

- dropping rows;
- dropping features;
- mean imputation;
- median imputation;
- most-frequent imputation;
- explicit "Unknown" category.

We should choose a strategy based on the feature and the meaning of the missingness.

### 7.1 Demonstrate `dropna()`

We demonstrate it, but we will not use it as our final strategy because the dataset is very small.

In [ ]:
print("Original rows:", len(df))
print("Rows after dropna():", len(df.dropna()))

df.dropna()

### 7.2 Demonstrate `fillna()`

Pandas can fill missing values directly.

This is useful for understanding imputation, although later we will use Scikit-Learn so the imputation can be fitted on training data only.

In [ ]:
fill_demo = df.copy()

fill_demo["Age"] = fill_demo["Age"].fillna(fill_demo["Age"].median())
fill_demo["Salary"] = fill_demo["Salary"].fillna(fill_demo["Salary"].median())

fill_demo

## Part 8 – Outlier inspection

An unusual value is not automatically an error.

We first inspect distributions before deciding whether an observation should be changed or removed.

In [ ]:
df[["Age", "Salary"]].describe()

### Boxplot

A boxplot is a simple visual tool for spotting observations that lie far from most of the data.

In [ ]:
df[["Age", "Salary"]].plot(kind="box")
plt.title("Numeric features before preprocessing")
plt.show()

### IQR demonstration

The interquartile range (IQR) can be used as a simple diagnostic rule:

```
IQR = Q3 - Q1
Lower bound = Q1 - 1.5 × IQR
Upper bound = Q3 + 1.5 × IQR
```

This identifies possible outliers; it does **not** automatically mean they should be deleted.

In [ ]:
salary_non_missing = df["Salary"].dropna()

q1 = salary_non_missing.quantile(0.25)
q3 = salary_non_missing.quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)

possible_outliers = df[
    (df["Salary"] < lower) | (df["Salary"] > upper)
]

possible_outliers

## Part 9 – Data integration

Data integration combines information from multiple sources.

The key lesson is that integration is not simply concatenation: identifiers, meanings, units, and categories must be compatible.

Below is a small example using `merge()`.

In [ ]:
people = pd.DataFrame({
    "Person_ID": [1, 2, 3],
    "Name": ["Ali", "Sara", "Omar"],
    "Department_ID": [10, 20, 10]
})

departments = pd.DataFrame({
    "Department_ID": [10, 20],
    "Department": ["IT", "Business"]
})

print("People:")
display(people)

print("Departments:")
display(departments)

In [ ]:
integrated = people.merge(
    departments,
    on="Department_ID",
    how="left"
)

integrated

## Part 10 – Data transformation: aggregation

Aggregation summarizes records at a level appropriate to the analysis.

For example, we can summarize salary by city.

In [ ]:
city_summary = (
    df.groupby("City", dropna=False)
      .agg(
          Number_of_People=("City", "size"),
          Mean_Salary=("Salary", "mean"),
          Median_Age=("Age", "median")
      )
      .reset_index()
)

city_summary

## Part 11 – Discretization

Discretization converts a continuous variable into intervals or categories.

Here we convert Age into age groups using `pd.cut()`.

In [ ]:
age_demo = df.copy()

age_demo["Age_Group"] = pd.cut(
    age_demo["Age"],
    bins=[0, 30, 40, 50, np.inf],
    labels=["Young", "Adult", "Mature", "Older"],
    include_lowest=True
)

age_demo[["Age", "Age_Group"]]

### Discussion

Discretization can make a variable easier to interpret, but it also loses information inside each interval.

## Part 12 – Feature selection vs. feature extraction

Lecture 3 introduces the distinction:

- **Feature selection** keeps some of the original variables.
- **Feature extraction** creates new variables from the originals.
- **PCA** is a feature-extraction method and will be studied in depth later.

For now, we only demonstrate feature selection.

In [ ]:
selected_features = df[["Age", "Salary"]]

selected_features.head()

## Part 13 – Separate predictors and target

Now we move from general data understanding into the machine-learning workflow.

In [ ]:
X = df.drop(columns="Purchased")
y = df["Purchased"]

print("Features:")
display(X.head())

print("Target:")
display(y.head())

## Part 14 – Split before fitting preprocessing

Any preprocessing step that **learns parameters from data** must be fitted using the training set only.

Examples include:

- mean/median imputation;
- minimum and maximum values;
- mean and standard deviation;
- learned categories.

If these are learned from the complete dataset before splitting, information from the test set leaks into training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Part 15 – Imputation with Scikit-Learn

Now we use `SimpleImputer` in the leakage-safe workflow.

We fit on training data and only transform the test data.

In [ ]:
from sklearn.impute import SimpleImputer

numeric_features = ["Age", "Salary"]

num_imputer = SimpleImputer(strategy="median")

X_train_num = num_imputer.fit_transform(X_train[numeric_features])
X_test_num = num_imputer.transform(X_test[numeric_features])

print("Medians learned from training data:")
print(num_imputer.statistics_)

## Part 16 – Categorical encoding

`City` is nominal: there is no natural order among city names.

Therefore one-hot encoding is appropriate.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_city = encoder.fit_transform(X_train[["City"]])
X_test_city = encoder.transform(X_test[["City"]])

print("Learned categories:")
print(encoder.categories_)

print("\nEncoded training data:")
print(X_train_city)

## Part 17 – Target encoding

The target `Purchased` contains two class labels: Yes and No.

Encoding the **target** is different from encoding a nominal input feature.

In [ ]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()

y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

print("Classes:", target_encoder.classes_)
print("Encoded training target:", y_train_encoded)

## Part 18 – Normalization vs. standardization

### Min–Max normalization

Maps values to a fixed range, commonly 0–1.

### Standardization

Centers a variable around 0 and scales using its standard deviation.

We compare both using training data only.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Impute training data first
train_num_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index
)

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

standardized = standard_scaler.fit_transform(train_num_imputed)
normalized = minmax_scaler.fit_transform(train_num_imputed)

print("Original / imputed:")
display(train_num_imputed)

print("Standardized:")
display(pd.DataFrame(
    standardized,
    columns=numeric_features,
    index=X_train.index
))

print("Normalized:")
display(pd.DataFrame(
    normalized,
    columns=numeric_features,
    index=X_train.index
))

### Verification

After standardization, the training feature means should be approximately 0.

After Min–Max normalization, values should lie approximately between 0 and 1.

In [ ]:
standardized_df = pd.DataFrame(
    standardized,
    columns=numeric_features
)

normalized_df = pd.DataFrame(
    normalized,
    columns=numeric_features
)

print("Standardized means:")
print(standardized_df.mean())

print("\nStandardized standard deviations:")
print(standardized_df.std(ddof=0))

print("\nNormalized minimums:")
print(normalized_df.min())

print("\nNormalized maximums:")
print(normalized_df.max())

## Part 19 – Complete preprocessing with `ColumnTransformer`

We now combine the earlier steps into one reproducible object.

Numeric features:

**median imputation → standardization**

Categorical feature:

**most-frequent imputation → one-hot encoding**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ["Age", "Salary"]
categorical_features = ["City"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

preprocessor

## Part 20 – Fit on training data and transform test data

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nFinal feature names:")
print(preprocessor.get_feature_names_out())

## Part 21 – End-to-end pipeline

We include a simple classifier only to demonstrate the full workflow.

The goal here is **not** to evaluate Logistic Regression in depth.

The dataset is far too small for meaningful performance conclusions.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression())
])

model.fit(X_train, y_train)

print("Training accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

## Part 22 – Data leakage: wrong vs. correct

### Wrong

```python
imputer.fit(full_dataset)
scaler.fit(full_dataset)
train_test_split(...)
```

The test data influenced the preprocessing parameters.

### Correct

```python
train_test_split(...)
imputer.fit(X_train)
imputer.transform(X_test)
```

The test set remains unseen during fitting.

## Student exercises

### Exercise 1 – Inspection

Using the original DataFrame:

1. show the first 3 rows;
2. show the last 3 rows;
3. print the shape;
4. run `info()`;
5. calculate missing-value counts;
6. calculate missing percentages;
7. print unique cities;
8. count each target class.

### Exercise 2 – Cleaning

Create a copy of the dataset and:

1. add one duplicate row;
2. detect it;
3. remove it;
4. change one city to lowercase;
5. add spaces around another city;
6. normalize the strings.

### Exercise 3 – Missing values

Compare:

- `dropna()`
- mean imputation
- median imputation

Explain what each method changes.

### Exercise 4 – Transformation

1. aggregate mean salary by city;
2. create age groups;
3. compare MinMaxScaler and StandardScaler.

### Exercise 5 – Leakage

Write:

- one incorrect preprocessing workflow that leaks information;
- one correct workflow.

Explain why the first is invalid.

## Challenge

Choose another small tabular dataset and repeat this complete workflow:

**Load → Inspect → Diagnose → Clean → Transform → Encode → Split → Pipeline**

Your notebook should explicitly use and explain:

- `head()`
- `info()`
- `shape`
- `dtypes`
- `describe()`
- `isna()`
- `value_counts()`
- `duplicated()`
- one missing-value strategy
- one scaling method
- one categorical encoding method
- train/test split
- a leakage-safe pipeline

## Reflection

1. Why should `head()` and `info()` be used immediately after loading data?
2. What is the difference between `isna()` and `isna().sum()`?
3. Why is `value_counts()` useful before encoding categorical data?
4. Why does an outlier require investigation rather than automatic deletion?
5. What is the difference between cleaning and transformation?
6. What is data integration?
7. What is the difference between feature selection and feature extraction?
8. Why must scaling parameters be learned only from the training set?
9. Why is one-hot encoding appropriate for City?
10. Why is a pipeline useful for reproducibility?

---

**ML6402 – Machine Learning**  
**Instructor: Dr. Ahmed Sabbah**